In [20]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
import pandas as pd
import muon as mu
import scanpy as sc
import scirpy as ir
np.random.seed(42)
import random
random.seed(42)
import sys
sys.path.append(r"E:\Python code\Machine learning\JupyterNote\Bio_CRC\Data processing\functions")
import mdata_utils 
import TCR_embedings
from sklearn.preprocessing import StandardScaler
from scipy.sparse import issparse

In [21]:
df = pd.read_csv(r".\Raw features\tcr_ex_clone.csv")
df.head(5)

,clone_id,X_VDJ_1_cdr3_aa_atchley_0,X_VDJ_1_cdr3_aa_atchley_1,X_VDJ_1_cdr3_aa_atchley_2,X_VDJ_1_cdr3_aa_atchley_3,X_VDJ_1_cdr3_aa_atchley_4,X_VDJ_1_cdr3_aa_atchley_5,X_VDJ_1_cdr3_aa_atchley_6,X_VDJ_1_cdr3_aa_atchley_7,X_VDJ_1_cdr3_aa_atchley_8,...,X_VJ_1_cdr3_aa_composition_13,X_VJ_1_cdr3_aa_composition_14,X_VJ_1_cdr3_aa_composition_15,X_VJ_1_cdr3_aa_composition_16,X_VJ_1_cdr3_aa_composition_17,X_VJ_1_cdr3_aa_composition_18,X_VJ_1_cdr3_aa_composition_19,VDJ_1_cdr3_aa_length,VJ_1_cdr3_aa_length,ex_trend
0,0,0.235605,-0.279393,0.271942,0.278817,0.264461,0.137911,0.285695,-0.365258,-0.025915,...,0.887956,1.536931,1.390758,-0.245931,-0.940186,-0.225208,-0.827658,0.484995,1.877972,0.007753
1,1,0.235605,-0.279393,0.271942,0.278817,0.264461,0.137911,0.285695,-0.365258,-0.025915,...,-0.662635,0.477979,0.628000,-0.172986,-0.940186,-0.225208,-0.827658,-0.520839,1.167852,0.014659
2,10,0.235605,-0.279393,0.271942,0.278817,0.264461,0.137911,0.285695,-0.365258,-0.025915,...,-0.662635,-0.743889,-0.444630,-0.309758,1.138391,3.339770,0.359500,0.484995,2.588093,-0.019912
3,100,0.235605,-0.279393,0.271942,0.278817,0.264461,0.137911,0.285695,-0.365258,-0.025915,...,-0.662635,-0.743889,-1.278897,0.009377,1.831250,-0.225208,-0.827658,0.484995,-0.252390,0.015591
4,1000,0.235605,-0.279393,0.271942,0.278817,0.264461,0.137911,0.285695,-0.365258,-0.025915,...,-0.662635,0.571969,-1.278897,-0.088819,0.338939,-0.225208,0.633460,-1.023756,0.457731,-0.010393


In [22]:
sig_LF_index = [5,8,11,27]

In [23]:
subtype = ["CD8_Teff"]
clone_status = ["Tumor_single"]


In [24]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from scipy.stats import pearsonr

df_sub = df

feature_cols = df.columns[1:-1]
X = df_sub[feature_cols].values
y = df_sub['ex_trend'].values

print(f"Subset: {len(df_sub)} cells  |  Features: {len(feature_cols)}")
print(f"ex_trend range: [{y.min():.4f}, {y.max():.4f}]  mean={y.mean():.4f}")

Subset: 2349 cells  |  Features: 242
ex_trend range: [-0.1486, 0.2715]  mean=0.0154


In [25]:
## Linear regression with ALL features (col 5+) ##
scaler_all = StandardScaler()
X_scaled = scaler_all.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

# linear regression
# lr_all = LinearRegression()

# XGboost
from xgboost import XGBRegressor
lr_all = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

lr_all.fit(X_train, y_train)

y_pred = lr_all.predict(X_test)

r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
corr, pval = pearsonr(y_test, y_pred)

print("=" * 60)
print(f"Linear Regression — ALL features ({len(feature_cols)})")
print(f"Train: {len(X_train)}  |  Test: {len(X_test)}")
print(f"R²: {r2:.4f}   MSE: {mse:.6f}   Pearson r: {corr:.4f} (p={pval:.4g})")
print("=" * 60)

Linear Regression — ALL features (242)
Train: 1879  |  Test: 470
R²: -0.0592   MSE: 0.001439   Pearson r: 0.0688 (p=0.1361)


In [26]:
import os

slide_dir = r"E:\Biology\scRNA\Tumor\human\gex+tcr\GSE139555 Pan-cancer\SLIDE analysis\tcr_exh_clone\0.5_0.4_out"

slide_features = []
for idx in sig_LF_index:
    fpath = os.path.join(slide_dir, f"feature_list_Z{idx}.txt")
    fl = pd.read_csv(fpath, sep="\t")
    names = fl['names'].dropna().tolist()
    names = [n for n in names if n != "NA"]
    slide_features.extend(names)
    print(f"Z{idx}: {len(names)} features")

slide_features = sorted(set(slide_features))
valid_slide = [f for f in slide_features if f in feature_cols]
missing = [f for f in slide_features if f not in feature_cols]

print(f"\nTotal unique SLIDE features: {len(slide_features)}")
print(f"Matched in df: {len(valid_slide)}")
if missing:
    print(f"Missing from df: {missing}")

Z5: 8 features
Z8: 10 features
Z11: 13 features
Z27: 12 features

Total unique SLIDE features: 38
Matched in df: 38


In [30]:
## Linear regression with SLIDE-selected features ##
X_slide = df_sub[valid_slide].values

scaler_slide = StandardScaler()
X_slide_scaled = scaler_slide.fit_transform(X_slide)

X_tr_s, X_te_s, y_tr_s, y_te_s = train_test_split(
    X_slide_scaled, y, test_size=0.2, random_state=42
)

# lr_slide = LinearRegression()
lr_slide = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

lr_slide.fit(X_tr_s, y_tr_s)

y_pred_s = lr_slide.predict(X_te_s)

r2_s = r2_score(y_te_s, y_pred_s)
mse_s = mean_squared_error(y_te_s, y_pred_s)
corr_s, pval_s = pearsonr(y_te_s, y_pred_s)

print("=" * 60)
print(f"Linear Regression — SLIDE features ({len(valid_slide)})")
print(f"Train: {len(X_tr_s)}  |  Test: {len(X_te_s)}")
print(f"R²: {r2_s:.4f}   MSE: {mse_s:.6f}   Pearson r: {corr_s:.4f} (p={pval_s:.4g})")
print("=" * 60)

print(f"\n--- Comparison ---")
print(f"ALL features  ({len(feature_cols):>3d}):  R²={r2:.4f}  MSE={mse:.6f}  r={corr:.4f}")
print(f"SLIDE features ({len(valid_slide):>3d}):  R²={r2_s:.4f}  MSE={mse_s:.6f}  r={corr_s:.4f}")

Linear Regression — SLIDE features (38)
Train: 1879  |  Test: 470
R²: -0.0820   MSE: 0.001470   Pearson r: 0.0714 (p=0.1224)

--- Comparison ---
ALL features  (242):  R²=-0.0592  MSE=0.001439  r=0.0688
SLIDE features ( 38):  R²=-0.0820  MSE=0.001470  r=0.0714


In [31]:
aa

NameError: name 'aa' is not defined

## Classification (binarized MANAscore)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score
from imblearn.over_sampling import RandomOverSampler

threshold = np.median(y)
y_cls = (y >= threshold).astype(int)

print(f"MANAscore threshold (median): {threshold:.4f}")
print(f"Class distribution: 0(low)={np.sum(y_cls==0)}, 1(high)={np.sum(y_cls==1)}")

MANAscore threshold (median): 0.0077
Class distribution: 0(low)=1174, 1(high)=1175


In [ ]:
## Logistic regression with ALL features (col 5+) ##
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    scaler_all.fit_transform(X), y_cls, test_size=0.2, random_state=42, stratify=y_cls
)

ros_c = RandomOverSampler(random_state=42)
X_train_c_bal, y_train_c_bal = ros_c.fit_resample(X_train_c, y_train_c)

clf_all = LogisticRegression(max_iter=1000, random_state=42)
clf_all.fit(X_train_c_bal, y_train_c_bal)

y_pred_c = clf_all.predict(X_test_c)
y_prob_c = clf_all.predict_proba(X_test_c)[:, 1]

acc_c = accuracy_score(y_test_c, y_pred_c)
try:
    auc_c = roc_auc_score(y_test_c, y_prob_c)
except ValueError:
    auc_c = np.nan

print("=" * 60)
print(f"Logistic Regression — ALL features ({len(feature_cols)})")
print(f"Train(balanced): {len(X_train_c_bal)}  |  Test: {len(X_test_c)}")
print(f"Accuracy: {acc_c:.4f}   ROC AUC: {auc_c:.4f}")
print("=" * 60)
print(classification_report(y_test_c, y_pred_c,
                            target_names=["MANA_low", "MANA_high"],
                            zero_division=0))

Logistic Regression — ALL features (242)
Train(balanced): 1880  |  Test: 470
Accuracy: 0.5085   ROC AUC: 0.5076
              precision    recall  f1-score   support

    MANA_low       0.51      0.51      0.51       235
   MANA_high       0.51      0.51      0.51       235

    accuracy                           0.51       470
   macro avg       0.51      0.51      0.51       470
weighted avg       0.51      0.51      0.51       470



In [ ]:
## Logistic regression with SLIDE-selected features ##
X_tr_sc, X_te_sc, y_tr_sc, y_te_sc = train_test_split(
    scaler_slide.fit_transform(df_sub[valid_slide].values), y_cls,
    test_size=0.2, random_state=42, stratify=y_cls
)

ros_sc = RandomOverSampler(random_state=42)
X_tr_sc_bal, y_tr_sc_bal = ros_sc.fit_resample(X_tr_sc, y_tr_sc)

clf_slide = LogisticRegression(max_iter=1000, random_state=42)
clf_slide.fit(X_tr_sc_bal, y_tr_sc_bal)

y_pred_sc = clf_slide.predict(X_te_sc)
y_prob_sc = clf_slide.predict_proba(X_te_sc)[:, 1]

acc_sc = accuracy_score(y_te_sc, y_pred_sc)
try:
    auc_sc = roc_auc_score(y_te_sc, y_prob_sc)
except ValueError:
    auc_sc = np.nan

print("=" * 60)
print(f"Logistic Regression — SLIDE features ({len(valid_slide)})")
print(f"Train(balanced): {len(X_tr_sc_bal)}  |  Test: {len(X_te_sc)}")
print(f"Accuracy: {acc_sc:.4f}   ROC AUC: {auc_sc:.4f}")
print("=" * 60)
print(classification_report(y_te_sc, y_pred_sc,
                            target_names=["MANA_low", "MANA_high"],
                            zero_division=0))

print(f"\n--- Classification Comparison ---")
print(f"ALL features  ({len(feature_cols):>3d}):  Acc={acc_c:.4f}  AUC={auc_c:.4f}")
print(f"SLIDE features ({len(valid_slide):>3d}):  Acc={acc_sc:.4f}  AUC={auc_sc:.4f}")

Logistic Regression — SLIDE features (38)
Train(balanced): 1880  |  Test: 470
Accuracy: 0.5426   ROC AUC: 0.5406
              precision    recall  f1-score   support

    MANA_low       0.55      0.51      0.53       235
   MANA_high       0.54      0.57      0.56       235

    accuracy                           0.54       470
   macro avg       0.54      0.54      0.54       470
weighted avg       0.54      0.54      0.54       470


--- Classification Comparison ---
ALL features  (242):  Acc=0.5085  AUC=0.5076
SLIDE features ( 38):  Acc=0.5426  AUC=0.5406
